# 12 — Particle tracking vs. July 2025 drifter deploys

**Goal:** reproduce the 12 Stokes-drifter deploys from the July 2025 field campaign with a Lagrangian particle tracking run driven by the v02 (or later) D-Flow FM velocity field, then score skill per deploy.

**Data:** `polished_data2.csv` — columns: `Data Date; Sea Surface; x_coord; y_coord; source; deploy; Distance; Time step; Velocity; deltaX; deltaY; quadrante; Direction; Direction N; ...`. Coordinates are UTM Zone 32N (EPSG:32632). Rows with `deploy=altri` are between-deploy drift / anomalies — excluded here.

**Approach** (chosen after literature review — see `memory/project_residence_time.md` for the residence-time sibling task):
1. Parse and QC drifter CSV (Italian locale, semicolon separator, comma decimals, mixed date formats)
2. Group by `(deploy, source)` → one drifter track each; first valid row = release (t₀, x₀, y₀)
3. Convert UTM32N → WGS84 to match v02 mesh CRS
4. Run particle tracking **offline via OpenDrift** reading `Stagnone_dxy01_15m_map.nc` (DIMR-native FM particle module is still immature in DFM 1.2.184 — OpenDrift is the pragmatic path)
5. Score trajectory skill per deploy: endpoint separation, path length ratio, mean heading bias

**Status:** prep + visualization ready; the OpenDrift execution cell is gated on v02 output.

## 1. Imports and paths

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

project_root = Path(r'F:\StagnoneDT')
raw_dir = project_root / 'data' / 'raw' / 'insitu'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

drifter_csv = raw_dir / 'drifters_Jul2025_polished.csv'
v02_map = project_root / 'model' / 'dflowfm_v02' / 'output' / 'Stagnone_dxy01_15m_map.nc'

print(f'Drifter CSV exists: {drifter_csv.exists()}  ({drifter_csv})')
print(f'v02 map file exists: {v02_map.exists()}  ({v02_map})')

## 2. Parse drifter CSV

Handles: UTF-8 BOM, semicolon delimiter, comma decimals, and two date formats present in the file (most rows `MM/DD/YYYY HH:MM`; a handful of QC-flagged rows use `DD/MM/YYYY HH:MM:SS`). Campaign spans **8–9 July 2025**.

Action: load raw, parse dates robustly, drop rows without coords, filter `deploy ∈ {1..12}` (drop `altri`).

In [ ]:
raw = pd.read_csv(
    drifter_csv,
    sep=';',
    encoding='utf-8-sig',
    decimal=',',
    engine='python',
)
# strip trailing empty columns from the messy Excel export
raw = raw.loc[:, ~raw.columns.str.match(r'^Unnamed')]
raw.columns = [c.strip() for c in raw.columns]
print(f'Raw rows: {len(raw)}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

In [ ]:
def parse_date(s):
    """Parse either MM/DD/YYYY HH:MM or DD/MM/YYYY HH:MM:SS. Returns NaT on failure."""
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    # rows with seconds use DD/MM/YYYY
    if s.count(':') == 2:
        return pd.to_datetime(s, format='%d/%m/%Y %H:%M:%S', errors='coerce')
    # main rows: MM/DD/YYYY HH:MM
    return pd.to_datetime(s, format='%m/%d/%Y %H:%M', errors='coerce')

raw['time'] = raw['Data Date'].apply(parse_date)

# Sanity: campaign was 8-9 July 2025
print('Time range:', raw['time'].min(), '→', raw['time'].max())
print('Rows with unparsed time:', raw['time'].isna().sum())

In [ ]:
# Select and clean
df = raw[['time', 'x_coord', 'y_coord', 'source', 'deploy', 'Velocity [m/s]', 'Direction N [\u00ba]']].copy()
df.columns = ['time', 'x_utm', 'y_utm', 'source', 'deploy', 'v_ms', 'dir_deg']

# Drop rows without time or coords
df = df.dropna(subset=['time', 'x_utm', 'y_utm']).reset_index(drop=True)

# Keep only numbered deploys 1-12 (drop 'altri' and any other non-numeric)
df['deploy'] = pd.to_numeric(df['deploy'], errors='coerce')
df = df.dropna(subset=['deploy'])
df['deploy'] = df['deploy'].astype(int)
df = df[df['deploy'].between(1, 12)].reset_index(drop=True)

print(f'After filtering: {len(df)} rows across deploys {sorted(df.deploy.unique())}')
df.groupby('deploy').agg(
    n_points=('time', 'size'),
    n_drifters=('source', 'nunique'),
    t_start=('time', 'min'),
    t_end=('time', 'max'),
)

## 3. Convert UTM32N → WGS84

The v02 mesh is in WGS84 (EPSG:4326). Drifter coords are UTM Zone 32N (EPSG:32632 — standard for Sicily).
Quick sanity check: transformed coords should land near `lon ~ 12.4–12.5°E, lat ~ 37.8–37.9°N`.

In [ ]:
transformer = Transformer.from_crs('EPSG:32632', 'EPSG:4326', always_xy=True)
df['lon'], df['lat'] = transformer.transform(df['x_utm'].values, df['y_utm'].values)

print(f'Lon range: {df.lon.min():.4f} .. {df.lon.max():.4f}')
print(f'Lat range: {df.lat.min():.4f} .. {df.lat.max():.4f}')
# Expected: ~12.44-12.49 E, ~37.84-37.90 N for Stagnone

## 4. Per-deploy release points

For each `(deploy, source)` we take the first row in time as the release point. These feed the Lagrangian tracker as `t₀, x₀, y₀` seeds.

In [ ]:
df = df.sort_values(['deploy', 'source', 'time']).reset_index(drop=True)
releases = df.groupby(['deploy', 'source'], as_index=False).first()[
    ['deploy', 'source', 'time', 'lon', 'lat']
]
releases.columns = ['deploy', 'drifter_id', 't0', 'lon0', 'lat0']

print(f'Total drifter seeds: {len(releases)}')
releases

In [ ]:
# Save release seeds and cleaned tracks for downstream use
releases_path = processed_dir / 'drifter_releases_Jul2025.csv'
tracks_path = processed_dir / 'drifter_tracks_Jul2025.csv'
releases.to_csv(releases_path, index=False)
df[['deploy', 'source', 'time', 'lon', 'lat', 'x_utm', 'y_utm', 'v_ms', 'dir_deg']].to_csv(tracks_path, index=False)
print(f'Wrote:\n  {releases_path}\n  {tracks_path}')

## 5. Visualize observed tracks

Color by deploy number so spatial clustering is obvious. The campaign had drifters released at various points inside and across the inlets — this is our ground truth for the Lagrangian run.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 11))
cmap = plt.get_cmap('tab20')
for i, (dep, g) in enumerate(df.groupby('deploy')):
    color = cmap(i % 20)
    for _, sg in g.groupby('source'):
        ax.plot(sg['lon'], sg['lat'], '-', color=color, alpha=0.6, lw=1)
    # release points
    seeds = releases[releases['deploy'] == dep]
    ax.scatter(seeds['lon0'], seeds['lat0'], color=color, s=45, marker='o',
               edgecolor='k', linewidth=0.6, label=f'D{dep} (n={len(seeds)})', zorder=5)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Stagnone drifter tracks — July 2025 (12 deploys)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.set_aspect(1 / np.cos(np.radians(37.87)))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Run Lagrangian tracking

**Option A — OpenDrift (recommended, offline).** Reads `_map.nc`, seeds at the release points above, integrates surface trajectories with user-set diffusivity. Install via `conda install -c conda-forge opendrift`. Pseudocode below; not executed until v02 output is validated.

**Option B — D-Flow FM built-in particle module** (re-run model with `PartFile` pointing to a seed file matching `releases`). More faithful (same numerics as circulation) but requires re-running the full simulation.

The OpenDrift skeleton below is the first pass; switch to Option B only if surface-only drift under-/over-shoots systematically.

In [ ]:
# OpenDrift skeleton — run once v02 validates
# ---------------------------------------------
# from opendrift.readers.reader_netCDF_CF_unstructured import Reader
# from opendrift.models.oceandrift import OceanDrift
#
# reader = Reader(str(v02_map))
# o = OceanDrift(loglevel=20)
# o.add_reader(reader)
# o.set_config('drift:horizontal_diffusivity', 0.1)  # m²/s, tune vs. observed spread
# o.set_config('drift:advection_scheme', 'runge-kutta4')
# o.set_config('general:coastline_action', 'previous')
#
# for _, row in releases.iterrows():
#     o.seed_elements(lon=row.lon0, lat=row.lat0, time=row.t0.to_pydatetime(),
#                     number=1, z=0,
#                     origin_marker=int(row.deploy * 100 + hash(row.drifter_id) % 100))
#
# t_end = df['time'].max().to_pydatetime()
# o.run(end_time=t_end, time_step=60, time_step_output=300,
#       outfile=str(processed_dir / 'opendrift_v02.nc'))

print('OpenDrift run is gated on v02 completion + validation. Code stub above ready to activate.')

## 7. Skill metrics (per deploy)

When the simulated trajectory file is available, compute per-drifter:
- **Endpoint separation:** great-circle distance between observed and simulated final positions.
- **Path length ratio:** sim_pathlen / obs_pathlen (1 = perfect).
- **Mean heading bias:** circular mean of (sim_heading − obs_heading) along the track.
- **Liu & Weisberg (2011) skill score** `s = 1 − 〈d〉/〈L〉` where d is separation and L is cumulative observed path — the standard trajectory metric in oceanography.

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    φ1, φ2 = np.radians(lat1), np.radians(lat2)
    dφ = np.radians(lat2 - lat1)
    dλ = np.radians(lon2 - lon1)
    a = np.sin(dφ/2)**2 + np.cos(φ1) * np.cos(φ2) * np.sin(dλ/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def liu_weisberg_skill(obs_lon, obs_lat, sim_lon, sim_lat):
    """Liu & Weisberg (2011) trajectory skill score. Expects arrays at matching times."""
    d = haversine_m(obs_lon, obs_lat, sim_lon, sim_lat)            # separation at each time
    step = haversine_m(obs_lon[:-1], obs_lat[:-1], obs_lon[1:], obs_lat[1:])
    L = np.cumsum(np.concatenate([[0.0], step]))                    # cumulative obs path length
    # time-integrated ratio; skip t=0 where L=0
    valid = L > 0
    if not valid.any():
        return np.nan
    c = (d[valid] / L[valid]).sum() / valid.sum()
    return max(0.0, 1.0 - c)

print('Skill functions ready. Apply once opendrift_v02.nc is produced.')

## 8. Next steps

1. Wait for v02 completion → confirm mean bias, drying, 3D structure are acceptable
2. Activate the OpenDrift cell with `v02_map` as reader
3. Tune horizontal diffusivity against observed spread (start at 0.1 m²/s)
4. Compute metrics per deploy and per drifter; plot obs vs sim track pairs
5. If systematic surface under-drift: the v02 wind-drift is too weak → revisit wind drag (`icdtyp`, Charnock coefficient) or Stokes drift from v03 SWAN coupling
6. Residence-time companion notebook (13) will use same v02 output but with an Eulerian passive tracer instead